# Create Views

## Technical Views
- Create these views based from `events_distance_type`
```
1. distance_km results
2. distance_mi results
3. time_hours results
```

## Analytical Views
- Create more extra high analytical views
```
4. vw_global_leaderboard
5. vw_seasonal_race_trends
6. vw_runner_demographics
```

## Marathon types
- In the gold layer, I want to create a view for the **marathon types** coming from the event_distance_type.

In [0]:
%sql
-- ============================================================================
-- Query: Event Distance Type Distribution Analysis
-- Purpose: Count the number of race results by event distance type to understand
--          the distribution of distance_km, distance_mi, and time_hours events
-- ============================================================================

SELECT
    e.event_distance_type,              -- Type of distance measurement (km, mi, hours)
    COUNT(*) AS row_count                -- Total number of race results per type
FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
GROUP BY e.event_distance_type
ORDER BY row_count DESC;

#### 1. `distance_km`
- Start with **kilometer-based** distance events:
- It filters Gold star schema to only event_distance_type = 'distance_km'. 
- So this view will represent: **All race result rows where the event distance is measured in kilometers.**
- I have 5,572,102 rows numbers of km based race events.
- Then save it as a reusable analytical view: `vw_distance_km_results`

In [0]:
%sql
-- ============================================================================
-- Query: Kilometer-based Events Detail View (TEST)
-- Purpose: Validate the structure and content for vw_distance_km_results
-- Grain: One row per athlete result in a kilometer-based event
-- ============================================================================

SELECT
    -- Result identification
    f.result_id,

    -- Race date information
    d.event_start_date AS race_date,
    d.year AS race_year,
    d.month_name AS race_month,
    d.day AS race_day,

    -- Event details
    e.event_name,
    e.event_distance_length AS race_distance_label,
    e.event_distance_value AS race_distance_value,
    e.event_distance_unit AS race_distance_unit,

    -- Athlete details
    a.athlete_id,

    CASE
        WHEN a.athlete_gender = 'M' THEN 'Male'
        WHEN a.athlete_gender = 'F' THEN 'Female'
        ELSE 'Other / Unknown'
    END AS gender,

    a.athlete_year_of_birth AS birth_year,

    CASE
        WHEN a.athlete_age_category LIKE '%U23' THEN 'Under 23'
        WHEN a.athlete_age_category LIKE 'M%' THEN CONCAT(REPLACE(a.athlete_age_category, 'M', ''), '+')
        WHEN a.athlete_age_category LIKE 'W%' THEN CONCAT(REPLACE(a.athlete_age_category, 'W', ''), '+')
        ELSE a.athlete_age_category
    END AS age_group,

    f.athlete_age_at_event AS athlete_age,

    -- Country
    c.country_name,

    -- Performance metrics
    f.event_number_of_finishers AS number_of_finishers,
    f.athlete_performance AS finish_time,
    f.athlete_performance_seconds AS finish_time_seconds,
    f.athlete_average_speed_kmh AS average_speed_kmh

FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
INNER JOIN marathos_catalog.gold.dim_athlete a
    ON f.athlete_id = a.athlete_id
INNER JOIN marathos_catalog.gold.dim_country c
    ON f.country_code_iso3 = c.country_code_iso3
INNER JOIN marathos_catalog.gold.dim_date d
    ON f.date_id = d.date_id

WHERE e.event_distance_type = 'distance_km'

LIMIT 100;

In [0]:
%sql
-- ============================================================================
-- Query: Kilometer-based Events Row Count Validation
-- Purpose: Verify the total number of race results for km-based events
--          Expected: 5,572,102 rows
-- ============================================================================

SELECT
    COUNT(*) AS distance_km_view_rows    -- Total count of km-based race results
FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
WHERE e.event_distance_type = 'distance_km';

## 2. `distance_mi`
- This is **miles-based** distance events:
- It filters Gold star schema to only event_distance_type = 'distance_mi'. 
- So this view will represent: **All race result rows where the event distance is measured in miles.**
- I have 725,223 rows numbers of km based race events.
- Then save it as a reusable analytical view: `vw_distance_mi_results`

In [0]:
%sql
-- ============================================================================
-- Query: Miles-based Events Detail View (TEST)
-- Purpose: Validate the structure and content for vw_distance_mi_results
-- Grain: One row per athlete result in a mile-based event
-- ============================================================================

SELECT
    -- Result identification
    f.result_id,

    -- Race date information
    d.event_start_date AS race_date,
    d.year AS race_year,
    d.month_name AS race_month,
    d.day AS race_day,

    -- Event details
    e.event_name,
    e.event_distance_length AS race_distance_label,
    e.event_distance_value AS race_distance_value,
    e.event_distance_unit AS race_distance_unit,

    -- Athlete details
    a.athlete_id,

    CASE
        WHEN a.athlete_gender = 'M' THEN 'Male'
        WHEN a.athlete_gender = 'F' THEN 'Female'
        ELSE 'Other / Unknown'
    END AS gender,

    a.athlete_year_of_birth AS birth_year,

    CASE
        WHEN a.athlete_age_category LIKE '%U23' THEN 'Under 23'
        WHEN a.athlete_age_category LIKE 'M%' THEN CONCAT(REPLACE(a.athlete_age_category, 'M', ''), '+')
        WHEN a.athlete_age_category LIKE 'W%' THEN CONCAT(REPLACE(a.athlete_age_category, 'W', ''), '+')
        ELSE a.athlete_age_category
    END AS age_group,

    f.athlete_age_at_event AS athlete_age,

    -- Country
    c.country_name,

    -- Performance metrics
    f.event_number_of_finishers AS number_of_finishers,
    f.athlete_performance AS finish_time,
    f.athlete_performance_seconds AS finish_time_seconds,
    f.athlete_average_speed_kmh AS average_speed_kmh

FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
INNER JOIN marathos_catalog.gold.dim_athlete a
    ON f.athlete_id = a.athlete_id
INNER JOIN marathos_catalog.gold.dim_country c
    ON f.country_code_iso3 = c.country_code_iso3
INNER JOIN marathos_catalog.gold.dim_date d
    ON f.date_id = d.date_id

-- Filter to only mile-based distance events
WHERE e.event_distance_type = 'distance_mi'

LIMIT 100;

In [0]:
%sql
-- ============================================================================
-- Query: Miles-based Events Row Count Validation
-- Purpose: Verify the total number of race results for mile-based events
--          Expected: 725,223 rows
-- ============================================================================

SELECT
    COUNT(*) AS distance_mi_view_rows    -- Total count of mile-based race results
FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
WHERE e.event_distance_type = 'distance_mi';

%md
## 3. `time_hours`
- This is **hours based** time events:
- It filters Gold star schema to only event_distance_type = 'time_hours'. 
- So this view will represent: **All race result rows where the event time is measured in hours**
- I have 473,755 rows numbers of km based race events.
- Then save it as a reusable analytical view: `vw_time_hours_results`

In [0]:
%sql
-- ============================================================================
-- Query: Time-based Events Detail View (TEST)
-- Purpose: Validate the structure and content for vw_time_hour_results.
--          Returns business-ready race results for fixed-time events measured
--          in hours, such as 6-hour, 12-hour, 24-hour, or 48-hour races.
-- Grain: One row per athlete result in a fixed-time event.
-- ============================================================================

SELECT
    -- Result identification
    f.result_id,

    -- Race date information
    d.event_start_date AS race_date,
    d.year AS race_year,
    d.month_name AS race_month,
    d.day AS race_day,

    -- Event details
    e.event_name,
    e.event_distance_length AS race_time_limit_label,
    e.event_distance_value AS race_time_limit_hours,
    e.event_distance_unit AS race_time_unit,

    -- Athlete details
    a.athlete_id,

    CASE
        WHEN a.athlete_gender = 'M' THEN 'Male'
        WHEN a.athlete_gender = 'F' THEN 'Female'
        ELSE 'Other / Unknown'
    END AS gender,

    a.athlete_year_of_birth AS birth_year,

    CASE
        WHEN a.athlete_age_category LIKE '%U23' THEN 'Under 23'
        WHEN a.athlete_age_category LIKE 'M%' THEN CONCAT(REPLACE(a.athlete_age_category, 'M', ''), '+')
        WHEN a.athlete_age_category LIKE 'W%' THEN CONCAT(REPLACE(a.athlete_age_category, 'W', ''), '+')
        ELSE a.athlete_age_category
    END AS age_group,

    f.athlete_age_at_event AS athlete_age,

    -- Country
    c.country_name,

    -- Performance metrics
    f.event_number_of_finishers AS number_of_finishers,
    f.athlete_performance AS distance_covered,
    f.athlete_performance_distance AS distance_covered_km,
    f.athlete_average_speed_kmh AS average_speed_kmh

FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
INNER JOIN marathos_catalog.gold.dim_athlete a
    ON f.athlete_id = a.athlete_id
INNER JOIN marathos_catalog.gold.dim_country c
    ON f.country_code_iso3 = c.country_code_iso3
INNER JOIN marathos_catalog.gold.dim_date d
    ON f.date_id = d.date_id

-- Filter to only fixed-time events measured in hours
WHERE e.event_distance_type = 'time_hours'

LIMIT 100;

In [0]:
%sql
-- ============================================================================
-- Query: Time-based Events Row Count Validation
-- Purpose: Verify the total number of race results for hour-based events
--          Expected: 473,755 rows
-- ============================================================================

SELECT
    COUNT(*) AS time_hour_view_rows      -- Total count of time-based race results
FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
WHERE e.event_distance_type = 'time_hours';

## Analytical Views for Business

## 4. `vw_global_leaderboard`
- Shows the Country-level participation and performance ranking.
    - Top countries by results
    - Top countries by athletes
    - Average speed by country

---
- total_athlete_results = number of athlete result records (There num of athlete race result records where the athlete country is ---.)
- unique_athletes       = number of unique athletes
- unique_events         = number of unique events
- average_speed_kmh     = average cleaned speed in km/h
- average_athlete_age   = average athlete age during the event


In [0]:
%sql
-- ============================================================================
-- Analytical View: Global Leaderboard by Country
-- Purpose: Country-level participation and performance leaderboard.
--
-- Grain:
-- One row per country.
--
-- Business value:
-- This view shows which countries have the highest marathon/ultramarathon
-- participation and summarizes their athlete volume, event coverage,
-- average speed, and average athlete age.
--
-- Metric definitions:
-- race_result_records = number of athlete result rows.
--                       One athlete can appear many times if they joined many events.
-- unique_athletes     = number of distinct athletes.
-- events_participated = number of distinct events where athletes from this country appear.
-- ============================================================================

SELECT
    -- Country
    c.country_name,
    c.country_code_iso3 AS country_code,

    -- Participation metrics
    COUNT(*) AS race_result_records,
    COUNT(DISTINCT f.athlete_id) AS unique_athletes,
    COUNT(DISTINCT f.event_id) AS events_participated,

    -- Performance and profile metrics
    ROUND(AVG(f.athlete_average_speed_kmh), 3) AS average_speed_kmh,
    ROUND(AVG(f.athlete_age_at_event), 1) AS average_athlete_age

FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_country c
    ON f.country_code_iso3 = c.country_code_iso3

GROUP BY
    c.country_name,
    c.country_code_iso3

ORDER BY
    race_result_records DESC;

## 5. `vw_seasonal_race_trends`
- Time-based race participation trends by year and month.
    - Participation over time
    - Races by year
    - Monthly/seasonal patterns
    - Added the country of venue

In [0]:
%sql
-- ============================================================================
-- Analytical View: Seasonal Race Trends
-- Purpose: Show monthly marathon and ultramarathon activity over time.
--
-- Grain:
-- One row per year and month.
--
-- Business value:
-- This view helps stakeholders understand how race participation, event volume,
-- athlete engagement, speed, and athlete age change over time.
--
-- Metric definitions:
-- athlete_entries = number of athlete result records.
--                   One athlete can appear multiple times if they raced multiple events.
-- unique_athletes = number of distinct athletes active during the month.
-- events_held     = number of distinct events represented during the month.
-- ============================================================================

SELECT
    -- Time period
    d.year AS race_year,
    d.month_name AS race_month,

    -- Participation and activity metrics
    COUNT(*) AS athlete_entries,
    COUNT(DISTINCT f.athlete_id) AS unique_athletes,
    COUNT(DISTINCT f.event_id) AS events_held,

    -- Performance and profile metrics
    ROUND(AVG(f.athlete_average_speed_kmh), 3) AS average_speed_kmh,
    ROUND(AVG(f.athlete_age_at_event), 1) AS average_athlete_age

FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_date d
    ON f.date_id = d.date_id

GROUP BY
    d.year,
    d.month,
    d.month_name

ORDER BY
    d.year,
    d.month;

## 5b `vw_seasonal_race_trends_by_country`

In [0]:
%sql
-- ============================================================================
-- Analytical View: Seasonal Race Trends by Country
-- Purpose: Show monthly marathon and ultramarathon activity over time by athlete country.
--
-- Grain:
-- One row per year, month, and athlete country.
--
-- Business value:
-- This view helps stakeholders understand how race participation, athlete volume,
-- event coverage, speed, and athlete age change over time across countries.
--
-- Important note:
-- country_name represents the athlete's country, not necessarily the country
-- where the event was held.
--
-- Metric definitions:
-- athlete_entries = number of athlete result records.
--                   One athlete can appear multiple times if they raced multiple events.
-- unique_athletes = number of distinct athletes from the country during the month.
-- events_held     = number of distinct events represented by athletes from that country.
-- ============================================================================

SELECT
    -- Time period
    d.year AS race_year,
    d.month_name AS race_month,

    -- Athlete country
    c.country_name,

    -- Participation and activity metrics
    COUNT(*) AS athlete_entries,
    COUNT(DISTINCT f.athlete_id) AS unique_athletes,
    COUNT(DISTINCT f.event_id) AS events_held,

    -- Performance and profile metrics
    ROUND(AVG(f.athlete_average_speed_kmh), 3) AS average_speed_kmh,
    ROUND(AVG(f.athlete_age_at_event), 1) AS average_athlete_age

FROM marathos_catalog.gold.fact_results f
INNER JOIN marathos_catalog.gold.dim_date d
    ON f.date_id = d.date_id
INNER JOIN marathos_catalog.gold.dim_country c
    ON f.country_code_iso3 = c.country_code_iso3

GROUP BY
    d.year,
    d.month,
    d.month_name,
    c.country_name

ORDER BY
    d.year,
    d.month,
    athlete_entries DESC;

## 6. `vw_runner_demographics`
- Athlete profile analysis by gender and age category.
    - Results by gender
    - Results by age category
    - Average speed by demographic group

----

- Which gender and age groups participate the most?
- Which groups have the highest average speed?
- What is the average age and number of unique athletes by demographic group?

---

```
U23  = Male or Female underage <23

M23  = Male, around age category 23+
M35  = Male, age 35+
M40  = Male, age 40+
M45  = Male, age 45+

W23  = Woman/Female, around age category 23+
W35  = Woman/Female, age 35+
W40  = Woman/Female, age 40+
```

In [0]:
%sql
-- ============================================================================
-- Analytical View: Runner Demographics
-- Purpose: Show participation and performance by runner segment.
--
-- Grain:
-- One row per gender and age group.
-- ============================================================================

WITH demographic_base AS (
    SELECT
        CASE
            WHEN a.athlete_gender = 'M' THEN 'Male'
            WHEN a.athlete_gender = 'F' THEN 'Female'
            ELSE 'Other / Unknown'
        END AS gender,

        CASE
            WHEN a.athlete_age_category LIKE '%U%' THEN CONCAT('Under ', regexp_extract(a.athlete_age_category, '[0-9]+', 0))
            WHEN a.athlete_age_category LIKE 'M%' THEN CONCAT(regexp_extract(a.athlete_age_category, '[0-9]+', 0), '+')
            WHEN a.athlete_age_category LIKE 'W%' THEN CONCAT(regexp_extract(a.athlete_age_category, '[0-9]+', 0), '+')
            ELSE a.athlete_age_category
        END AS age_group,

        CASE
            WHEN a.athlete_age_category LIKE '%U%' THEN CAST(regexp_extract(a.athlete_age_category, '[0-9]+', 0) AS INT) - 1
            WHEN a.athlete_age_category LIKE 'M%' THEN CAST(regexp_extract(a.athlete_age_category, '[0-9]+', 0) AS INT)
            WHEN a.athlete_age_category LIKE 'W%' THEN CAST(regexp_extract(a.athlete_age_category, '[0-9]+', 0) AS INT)
            ELSE 999
        END AS age_group_sort,

        f.result_id,
        f.athlete_id,
        f.event_id,
        f.athlete_average_speed_kmh,
        f.athlete_age_at_event

    FROM marathos_catalog.gold.fact_results f
    INNER JOIN marathos_catalog.gold.dim_athlete a
        ON f.athlete_id = a.athlete_id

    WHERE a.athlete_age_category IS NOT NULL
      AND LOWER(a.athlete_age_category) <> 'unknown'
),

demographic_summary AS (
    SELECT
        gender,
        age_group,
        age_group_sort,
        COUNT(*) AS athlete_entries,
        COUNT(DISTINCT athlete_id) AS unique_athletes,
        COUNT(DISTINCT event_id) AS events_participated,
        ROUND(AVG(athlete_average_speed_kmh), 3) AS average_speed_kmh,
        ROUND(AVG(athlete_age_at_event), 1) AS average_athlete_age
    FROM demographic_base
    GROUP BY
        gender,
        age_group,
        age_group_sort
)

SELECT
    gender,
    age_group,
    athlete_entries,
    unique_athletes,
    ROUND(athlete_entries / unique_athletes, 2) AS entries_per_athlete,
    ROUND(
        athlete_entries * 100.0 / SUM(athlete_entries) OVER (),
        2
    ) AS participation_share_percent,
    events_participated,
    average_speed_kmh,
    average_athlete_age

FROM demographic_summary

ORDER BY
    gender,
    age_group_sort;

#### Shows:
| Column                | Meaning                                                                                                                              |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------ |
| `gender`              | Readable gender label, such as Male, Female, or Other / Unknown.                                                                     |
| `age_group`           | Readable age group, such as Under 23, 35+, 40+, or 50+.                                                                              |
| `athlete_entries`     | Number of race result records for that runner segment. One athlete can appear more than once if they participated in multiple races. |
| `unique_athletes`     | Number of distinct athletes in that runner segment. Each athlete is counted once.                                                    |
| `average_speed_kmh`   | Average cleaned speed in kilometers per hour for that runner segment.                                                                |
| `average_athlete_age` | Average actual age of athletes in that segment at the time of the race.                                                              |


---

- Runner participation by gender
- Runner participation by age group
- Average speed by gender and age group
- Average athlete age by runner segment


#### Important note
- athlete_entries counts race result records.
- unique_athletes counts individual athletes.
- For example, if one athlete appears in five races, they count as: 5 athlete_entries and 1 unique_athlet


In [0]:
%sql
-- ============================================================================
-- Analytical View: Runner Demographics
-- Purpose: Show simple participation and performance by gender and age group.
--
-- Grain:
-- One row per gender and age group.
--
-- Business value:
-- This view helps stakeholders understand which runner groups participate most
-- and how average speed and age differ between groups.
--
-- Metric definitions:
-- athlete_entries = number of athlete race result records.
-- unique_athletes = number of distinct athletes.
-- ============================================================================

WITH demographic_base AS (
    SELECT
        CASE
            WHEN a.athlete_gender = 'M' THEN 'Male'
            WHEN a.athlete_gender = 'F' THEN 'Female'
            ELSE 'Other / Unknown'
        END AS gender,

        CASE
            WHEN a.athlete_age_category LIKE '%U%' THEN CONCAT('Under ', regexp_extract(a.athlete_age_category, '[0-9]+', 0))
            WHEN a.athlete_age_category LIKE 'M%' THEN CONCAT(regexp_extract(a.athlete_age_category, '[0-9]+', 0), '+')
            WHEN a.athlete_age_category LIKE 'W%' THEN CONCAT(regexp_extract(a.athlete_age_category, '[0-9]+', 0), '+')
            ELSE a.athlete_age_category
        END AS age_group,

        CASE
            WHEN a.athlete_age_category LIKE '%U%' THEN CAST(regexp_extract(a.athlete_age_category, '[0-9]+', 0) AS INT) - 1
            WHEN a.athlete_age_category LIKE 'M%' THEN CAST(regexp_extract(a.athlete_age_category, '[0-9]+', 0) AS INT)
            WHEN a.athlete_age_category LIKE 'W%' THEN CAST(regexp_extract(a.athlete_age_category, '[0-9]+', 0) AS INT)
            ELSE 999
        END AS age_group_sort,

        f.athlete_id,
        f.athlete_average_speed_kmh,
        f.athlete_age_at_event

    FROM marathos_catalog.gold.fact_results f
    INNER JOIN marathos_catalog.gold.dim_athlete a
        ON f.athlete_id = a.athlete_id

    WHERE a.athlete_age_category IS NOT NULL
      AND LOWER(a.athlete_age_category) <> 'unknown'
),

demographic_summary AS (
    SELECT
        gender,
        age_group,
        age_group_sort,
        COUNT(*) AS athlete_entries,
        COUNT(DISTINCT athlete_id) AS unique_athletes,
        ROUND(AVG(athlete_average_speed_kmh), 3) AS average_speed_kmh,
        ROUND(AVG(athlete_age_at_event), 1) AS average_athlete_age
    FROM demographic_base
    GROUP BY
        gender,
        age_group,
        age_group_sort
)

SELECT
    gender,
    age_group,
    athlete_entries,
    unique_athletes,
    average_speed_kmh,
    average_athlete_age

FROM demographic_summary

ORDER BY
    gender,
    age_group_sort;